In [267]:
import re
import numpy as np
import pandas as pd

In [ ]:
# load data
forms = pd.read_csv('final-paraczech-package/paraczech/paraczech_forms.csv')
lexis = pd.read_csv('final-paraczech-package/paraczech/paraczech_lexemes.csv')
exponents = pd.read_csv('input-resources/exponents.csv')

# transform rules for exponents to regular expressions
rules_dict = exponents.groupby('cell').apply(
    lambda df: list(zip(df['regex'], df['exponent']))
).to_dict()

In [269]:
# delete irrelevant features from the cell description
def clear_cell(cell):
    features = ['pstg', 'cmpg', 'sprg', 'posp', 'negp']
    for feature in features:
        cell = cell.replace('.' + feature, '')
    return cell

In [270]:
# find indeclinable
# based on nominative and instrumental in singular
def indeclinable(group):
    safe_pairs = [
        ('nom.sg', 'ins.sg'),                 # nouns
        ('n.nom.pl', 'n.gen.pl'),             # adjectives
        ('f.nom.pl', 'f.gen.pl'),             # adjectives
        ('m.inan.nom.pl', 'm.inan.gen.pl'),   # adjectives
        ('m.anim.nom.pl', 'm.anim.gen.pl')    # adjectives
    ]

    g = group.copy()

    for fst_tag, scn_tag in safe_pairs:
        fst = set(g.loc[g['clean_cell'] == fst_tag, 'orth_form'])
        scn = set(g.loc[g['clean_cell'] == scn_tag, 'orth_form'])

        if fst and scn:
            if fst == scn: return True

    return False


forms['clean_cell'] = forms['cell'].apply(clear_cell)
indeclinable_forms = forms.groupby('lexeme').filter(indeclinable)
forms['indeclinable'] = forms['form_id'].isin(indeclinable_forms['form_id'])
del forms['clean_cell']

In [ ]:
# find exponents
def extract_and_verify_suffix(word, cell, rules):
    cell = clear_cell(cell)

    if not rules.get(cell, False):
        return np.nan

    for pattern, expected in rules[cell]:
        match = re.search(pattern, str(word))
        if match:
            if str(expected) == '0' or str(word).endswith(str(expected)):
                return '#NONE#' if expected == '0' else expected

    return np.nan

forms['extracted_exponent'] = forms.apply(
    lambda row: extract_and_verify_suffix(row['orth_form'], row['cell'], rules_dict), 
    axis=1
)

forms.loc[forms['indeclinable'] == True, 'extracted_exponent'] = ''

In [ ]:
indeclinable_map = forms.groupby('lexeme')['indeclinable'].any()
indeclinable_df = indeclinable_map.reset_index()

lexis = lexis.merge(indeclinable_df, left_on='lexeme_id', right_on='lexeme', how='left')
lexis['indeclinable'] = lexis['indeclinable'].fillna(False)
lexis['indeclinable'] = lexis['indeclinable'].astype(bool)

del lexis['lexeme']
del forms['indeclinable']

In [ ]:
forms.to_csv('final-paraczech-package/paraczech/paraczech_forms_new.csv')
lexis.to_csv('final-paraczech-package/paraczech/paraczech_lexemes_new.csv')